# Chapter 4: Validation of reproducibility across technical and biological replicates
## Create AAV6-ML figures for Chapter 3 for report for internship in AG Grimm
Author: Kolja Hildenbrand

Created: 22/04/2026

Finished: ...

## 1. Packages

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
import matplotlib as plt
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from itertools import product
from matplotlib_venn import venn2
from scipy.stats import pearsonr, spearmanr, linregress
from matplotlib.patches import Patch


## 2. Preperation 
### 2.1. Create Paths

In [ ]:
#---------------------------------
#---------------------------------
#---------------------------------

#general Path

general_dir = Path('/Users/kollybook/Library/Mobile Documents/com~apple~CloudDocs/Kolja_Hildenbrand/Uni/Master_Infectious_Diseases/Grimm_internship/Report_Grimm/Data') # <----- Hier den Server Path anpassen
os.makedirs(general_dir, exist_ok=True)

#---------------------------------
#---------------------------------
#---------------------------------

# Path for NGS data
NGS_dir = general_dir/'NGS_data'
os.makedirs(NGS_dir, exist_ok=True)

# Path for tables
tables_dir = general_dir/'tables'
os.makedirs(tables_dir, exist_ok=True)

#Path for figures
figures_dir = general_dir/'figures'/'4_validation'
os.makedirs(figures_dir, exist_ok=True)

### 2.2. Define helper functions


In [ ]:
# read csv files
def read_csv_file (path, file_name):
    df = pd.read_csv(path / f"{file_name}.csv")
    return df

In [ ]:
# extract list information from table
def extract_info(table, *columns):
    results = []
    
    for col in columns:
        if col not in table.columns:
            raise ValueError(f"Column '{col}' not found in table")
        
        unique_vals = (
            table[col]
            .dropna()
            .unique()
            .tolist()
        )
        
        results.append(sorted(unique_vals))
    
    return tuple(results)

In [ ]:
# sort file names from df_long
def sort_file_names (name_list):
    FILE_NAMES = {
        "gDNA": {
            "heart": {"biological": [], "technical": []},
            "liver": {"biological": [], "technical": []}
        },
        "cDNA": {
            "heart": {"biological": [], "technical": []},
            "liver": {"biological": [], "technical": []}
        }
    }
    
    for name in name_list:
        parts = name.split("_")
        n_parts = len(parts)
    
        # ext type
        if name.startswith("gDNA"):
            dna = "gDNA"
        elif name.startswith("cDNA"):
            dna = "cDNA"
        else:
            continue
    
        # Tissue 
        tissue = parts[1]
    
        if tissue not in ["heart", "liver"]:
            continue  
    
        # bio or tech
        if n_parts == 3:
            category = "biological"
        elif n_parts == 4:
            category = "technical"
        else:
            continue
    
        FILE_NAMES[dna][tissue][category].append(name)

    return FILE_NAMES 

In [ ]:
# comare and corr two tables
def compare_two_tables(
    df1,
    df2,
    name1="sample_1",
    name2="sample_2",
    compare_cols=None,
    key_col="AA_sequence",
    log_scale_cols=None,
    add_diagonal=True,
    alpha=0.25,
    point_size=8,
    figsize_per_panel=(5.5, 5),
    save=False,
    output_path=None,
    file_name=None,
    show=True
):
    """
    Compare two tables across one or multiple numeric columns by merging on key_col.

    Parameters
    ----------
    df1, df2 : pd.DataFrame
        Tables to compare
    name1, name2 : str
        Names shown in axis labels
    compare_cols : list of str
        Numeric columns to compare, e.g. ["Log2_enrichment", "RPM"]
    key_col : str
        Merge key, default = "AA_sequence"
    log_scale_cols : list of str
        Columns that should be plotted on log-log axes, e.g. ["RPM"]
    add_diagonal : bool
        Whether to add y=x diagonal
    alpha : float
        Point transparency
    point_size : int
        Scatter point size
    figsize_per_panel : tuple
        Width, height per subplot
    save : bool
        Whether to save figure
    output_path : str or Path
        Save directory
    file_name : str
        Output filename
    show : bool
        Whether to show plot

    Returns
    -------
    merged : pd.DataFrame
        Merged comparison table
    corr_df : pd.DataFrame
        Correlation summary table
    """

    if compare_cols is None:
        compare_cols = ["Log2_enrichment", "RPM"]

    if log_scale_cols is None:
        log_scale_cols = []

    # keep only needed columns
    cols1 = [key_col] + compare_cols
    cols2 = [key_col] + compare_cols

    a = df1[cols1].copy()
    b = df2[cols2].copy()

    # rename value columns
    a = a.rename(columns={col: f"{col}_{name1}" for col in compare_cols})
    b = b.rename(columns={col: f"{col}_{name2}" for col in compare_cols})

    # merge
    merged = a.merge(b, on=key_col, how="inner")

    # correlation results
    results = []

    # figure layout
    n = len(compare_cols)
    ncols = min(2, n)
    nrows = math.ceil(n / ncols)

    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=(figsize_per_panel[0] * ncols, figsize_per_panel[1] * nrows)
    )

    # make axes iterable
    if n == 1:
        axes = [axes]
    else:
        axes = np.array(axes).reshape(-1)

    for i, col in enumerate(compare_cols):
        ax = axes[i]

        xcol = f"{col}_{name1}"
        ycol = f"{col}_{name2}"

        sub = merged[[xcol, ycol]].dropna().copy()

        # correlations
        if len(sub) > 1:
            pearson_r, pearson_p = pearsonr(sub[xcol], sub[ycol])
            spearman_r, spearman_p = spearmanr(sub[xcol], sub[ycol])
        else:
            pearson_r, pearson_p = np.nan, np.nan
            spearman_r, spearman_p = np.nan, np.nan

        results.append({
            "column": col,
            "pearson_r": pearson_r,
            "pearson_p": pearson_p,
            "spearman_r": spearman_r,
            "spearman_p": spearman_p,
            "n_points": len(sub)
        })

        # scatter
        sns.scatterplot(
            data=sub,
            x=xcol,
            y=ycol,
            s=point_size,
            alpha=alpha,
            ax=ax
        )

        # log scaling if requested
        if col in log_scale_cols:
            sub_pos = sub[(sub[xcol] > 0) & (sub[ycol] > 0)].copy()
            ax.clear()
            sns.scatterplot(
                data=sub_pos,
                x=xcol,
                y=ycol,
                s=point_size,
                alpha=alpha,
                ax=ax
            )
            ax.set_xscale("log")
            ax.set_yscale("log")
            plot_data = sub_pos
        else:
            plot_data = sub

        # diagonal
        if add_diagonal and len(plot_data) > 0:
            lims = [
                min(plot_data[xcol].min(), plot_data[ycol].min()),
                max(plot_data[xcol].max(), plot_data[ycol].max())
            ]
            ax.plot(lims, lims, "--", color="black", linewidth=1.2, alpha=0.6)
            ax.set_xlim(lims)
            ax.set_ylim(lims)

        ax.set_xlabel(f"{col} ({name1})")
        ax.set_ylabel(f"{col} ({name2})")
        ax.set_title(col)

        ax.text(
            0.05, 0.95,
            f"Pearson r = {pearson_r:.4f}\nSpearman ρ = {spearman_r:.4f}",
            transform=ax.transAxes,
            ha="left",
            va="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.85)
        )

        sns.despine(ax=ax)

    # remove unused axes
    for j in range(n, len(axes)):
        fig.delaxes(axes[j])

    plt.tight_layout()

    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        import os
        os.makedirs(output_path, exist_ok=True)

        if file_name is None:
            col_string = "_".join(compare_cols)
            file_name = f"compare_{name1}_vs_{name2}_{col_string}.png"

        fig.savefig(os.path.join(output_path, file_name), dpi=300, bbox_inches="tight")

    if show:
        plt.show()
    else:
        plt.close(fig)

    corr_df = pd.DataFrame(results)

    return merged, corr_df

## 3. Script Functions

### 3.1. Single scatter plot technical rep -> biological rep -> sex rep 

In [ ]:
def corr_plot_two_samples(
    df1,
    df2,
    sample_1,
    sample_2,
    value_col="Log2_enrichment",
    aa_col="AA_sequence",
    pseudo_col="Pseudo",
    n_pseudo=0,
    plot="scatter",
    title=True,
    figsize=(5, 5),
    alpha=0.2,
    point_size=5,
    gridsize=100,
    cmap="viridis",
    show_table=False,
    save=False,
    output_path=None,
    file_name=None
):
    """
    Plot the correlation between two samples / groups after merging on AA_sequence.

    This function is agnostic to whether the inputs represent:
    - biological replicates
    - technical replicates
    - pooled sex groups
    - any other pre-filtered sample tables

    Parameters
    ----------
    df1, df2 : pd.DataFrame
        Input tables for the two samples/groups to compare.
        Each must contain at least:
        [aa_col, value_col, pseudo_col]
    sample_1, sample_2 : str
        Names used for axis labels and plot title.
    value_col : str, default='Log2_enrichment'
        Numeric column to correlate.
    aa_col : str, default='AA_sequence'
        Column used to merge the two tables.
    pseudo_col : str, default='Pseudo'
        Column used for pseudo filtering.
    n_pseudo : int, default=0
        Maximum allowed pseudo value in both samples.
    plot : str, default='scatter'
        'scatter' or 'hexbin'
    title : bool, default=True
        Whether to show a title.
    figsize : tuple, default=(5, 5)
        Figure size.
    alpha : float, default=0.2
        Transparency for scatter points.
    point_size : int, default=5
        Point size for scatter plot.
    gridsize : int, default=100
        Hexbin grid size.
    cmap : str, default='viridis'
        Colormap for hexbin.
    show_table : bool, default=False
        Whether to display the merged filtered table.
    save : bool, default=False
        Whether to save the figure.
    output_path : str or Path, optional
        Save directory. Required if save=True.
    file_name : str, optional
        Output file name.

    Returns
    -------
    df_merge : pd.DataFrame
        Merged and filtered table used for plotting.
    r : float
        Pearson correlation coefficient.
    """

    # ----------------------------------------------------------
    # Keep only required columns and rename for merge
    # ----------------------------------------------------------
    d1 = df1[[aa_col, value_col, pseudo_col]].copy()
    d2 = df2[[aa_col, value_col, pseudo_col]].copy()

    d1 = d1.rename(columns={
        value_col: f"{value_col}_{sample_1}",
        pseudo_col: f"{pseudo_col}_{sample_1}"
    })

    d2 = d2.rename(columns={
        value_col: f"{value_col}_{sample_2}",
        pseudo_col: f"{pseudo_col}_{sample_2}"
    })

    # ----------------------------------------------------------
    # Merge on AA sequence
    # ----------------------------------------------------------
    df_merge = d1.merge(d2, on=aa_col, how="inner")

    x_col = f"{value_col}_{sample_1}"
    y_col = f"{value_col}_{sample_2}"
    p1_col = f"{pseudo_col}_{sample_1}"
    p2_col = f"{pseudo_col}_{sample_2}"

    # ----------------------------------------------------------
    # Filter by pseudo support
    # ----------------------------------------------------------
    df_merge = df_merge[
        (df_merge[p1_col] <= n_pseudo) &
        (df_merge[p2_col] <= n_pseudo)
    ].copy()

    if df_merge.empty:
        raise ValueError("No variants left after merging and pseudo filtering.")

    if show_table:
        display(df_merge)

    # ----------------------------------------------------------
    # Create figure
    # ----------------------------------------------------------
    fig, ax = plt.subplots(figsize=figsize)

    if plot == "scatter":
        ax.scatter(
            df_merge[x_col],
            df_merge[y_col],
            alpha=alpha,
            s=point_size
        )
    elif plot == "hexbin":
        ax.hexbin(
            df_merge[x_col],
            df_merge[y_col],
            gridsize=gridsize,
            cmap=cmap,
            mincnt=1
        )
    else:
        raise ValueError("plot must be 'scatter' or 'hexbin'")

    # ----------------------------------------------------------
    # Pearson correlation
    # ----------------------------------------------------------
    r, _ = pearsonr(df_merge[x_col], df_merge[y_col])

    ax.text(
        0.05, 0.95,
        f"r = {r:.2f}",
        transform=ax.transAxes,
        fontsize=12,
        verticalalignment="top"
    )

    # ----------------------------------------------------------
    # Diagonal line
    # ----------------------------------------------------------
    lims = [
        min(df_merge[x_col].min(), df_merge[y_col].min()),
        max(df_merge[x_col].max(), df_merge[y_col].max())
    ]
    ax.plot(lims, lims, "--", color="tab:blue")

    # ----------------------------------------------------------
    # Regression line
    # ----------------------------------------------------------
    x = df_merge[x_col]
    y = df_merge[y_col]

    res = linregress(x, y)
    x_vals = np.linspace(x.min(), x.max(), 100)

    ax.plot(
        x_vals,
        res.slope * x_vals + res.intercept,
        color="red",
        linewidth=2
    )

    # ----------------------------------------------------------
    # Labels and title
    # ----------------------------------------------------------
    ax.set_xlabel(f"{value_col.replace('_', ' ')} ({sample_1})")
    ax.set_ylabel(f"{value_col.replace('_', ' ')} ({sample_2})")

    if title:
        ax.set_title(f"Correlation between {sample_1} and {sample_2}\nPseudo max. {n_pseudo}")

    plt.tight_layout()

    # ----------------------------------------------------------
    # Save
    # ----------------------------------------------------------
    if save:
        if output_path is None:
            raise ValueError("Please provide output_path when save=True")
        os.makedirs(output_path, exist_ok=True)

        if file_name is None:
            file_name = f"{plot}_corr_{sample_1}_vs_{sample_2}_max_{n_pseudo}_pseudo.png"

        fig.savefig(os.path.join(output_path, file_name), dpi=300, bbox_inches="tight")

    plt.show()

    return df_merge, r

### 3.2. KDE plot one per tissue, comparison gDNA vs cDNA

### 3.3. Venn2 of overlap between top 10000 Log2_enrichment¶

### 3.4. Diagramm of Venn2 plot overlap between top 1 and all variants

### 3.5. ECDF Log2 distribution between samples

## 4. Load files and extract information

### 4.1. Load csv files

In [ ]:
%%time
#load long table
df_long = read_csv_file(tables_dir / 'summary', "df_long_combined_biological_technical_rep")

# load pooled table
df_pooled = read_csv_file(tables_dir / 'summary', "df_sample_pooled")

# load pooled sex table
df_pooled_sex = read_csv_file(tables_dir / 'summary', "df_sample_pooled_sex")

In [ ]:
print ('Long Table')
display (df_long)

print ('Pooled Table')
display (df_pooled)

print ('Pooled Table sex')
display (df_pooled_sex)

### 4.2. Extract information from table

In [ ]:
SAMPLE, EXT, TISSUE, SEX, MOUSE_ID = extract_info(
    df_long, 
    'Sample', 
    'Extraction_type',
    "Tissue", 
    'Sex', 
    'Mouse_ID'
)

In [ ]:
display (SAMPLE, EXT, TISSUE, SEX, MOUSE_ID)

In [ ]:
# Sort different file names in extraction and biological or technical Replicates
DICT_NAMES = sort_file_names (SAMPLE)
DICT_NAMES

### 4.3. Load tissue/ext specific librarys¶

In [ ]:
%%time
# Load tissue specific library 
dict_library = {}
for tissue in TISSUE:
    df = read_csv_file(tables_dir/tissue, f'df_{tissue}_input_library')
    dict_library[tissue] = df

# Load raw library  for corr with special library
df_raw_input =  read_csv_file(tables_dir, 'df_input_lib_raw')

## 5. Figures

### 5.1. Biological Scatterplots cDNA liver (m2 vs m1)

In [ ]:
df1 = df_long[
    df_long["Sample"] == "cDNA_liver_m2"
][["AA_sequence", "Pseudo", "Log2_enrichment"]].copy()

df2 = df_long[
    df_long["Sample"] == "cDNA_liver_m1"
][["AA_sequence", "Pseudo", "Log2_enrichment"]].copy()

merged, r = corr_plot_two_samples(
    df1=df1,
    df2=df2,
    sample_1="cDNA liver m2",
    sample_2="cDNA liver m1",
    value_col="Log2_enrichment",
    title=False,
    n_pseudo=1,
    plot="scatter",
    save=True,
    output_path=figures_dir/'1_single_scatter_plot'/'1_biological_rep',
    file_name='scatter_corr_cDNA_liver_m2_vs_cDNA_liver_m1_no_title.png'
)

### 5.2. Technical Scatterplots cDNA liver (m2 vs Pro)

In [ ]:
df1 = df_long[
    df_long["Sample"] == "cDNA_liver_m2"
][["AA_sequence", "Pseudo", "Log2_enrichment"]].copy()

df2 = df_long[
    df_long["Sample"] == "cDNA_liver_PCR_rep"
][["AA_sequence", "Pseudo", "Log2_enrichment"]].copy()

merged, r = corr_plot_two_samples(
    df1=df1,
    df2=df2,
    sample_1="cDNA liver m2",
    sample_2="cDNA liver PCR rep",
    value_col="Log2_enrichment",
    n_pseudo=1,
    title=False,
    plot="scatter",
    save=True,
    output_path=figures_dir/'1_single_scatter_plot'/'1_technical_rep',
    file_name='scatter_corr_cDNA_liver_m2_vs_cDNA_liver_m2_PCR_rep_no_title.png'
)


### 5.3.Corr Scatterplot of sex cDNA liver (male vs female)

In [ ]:
df1 = df_pooled_sex[
    (df_pooled_sex["Tissue"] == "liver") &
    (df_pooled_sex["Extraction_type"] == "cDNA") &
    (df_pooled_sex["Sex"] == "male")
][["AA_sequence", "n_samples_present", "Log2_enrichment_old"]].copy()
df1 = df1.rename(columns={'Log2_enrichment_old': 'Log2_enrichment'})

df2 = df_pooled_sex[
    (df_pooled_sex["Tissue"] == "liver") &
    (df_pooled_sex["Extraction_type"] == "cDNA") &
    (df_pooled_sex["Sex"] == "female")
][["AA_sequence", "n_samples_present", "Log2_enrichment_old"]].copy()
df2 = df2.rename(columns={'Log2_enrichment_old': 'Log2_enrichment'})

merged, r = corr_plot_two_samples(
    df1=df1,
    df2=df2,
    sample_1="male",
    sample_2="female",
    value_col="Log2_enrichment",
    pseudo_col='n_samples_present',
    n_pseudo=3,
    title=False,
    plot="scatter",
    save=True,
    output_path=figures_dir/'1_single_scatter_plot'/'3_sex_rep',
    file_name='scatter_corr_cDNA_liver_male_vs_cDNA_liver_female_no_title.png'
)

### 5.4. Diagramm of Venn2 plot overlap between top 1 and all variants

### 5.5. ECDF Log2_enrichment_gDNA_to_cDNA distribution between samples